## 1. Imports y configuración

In [1]:
import pandas as pd
from faker import Faker
import uuid
import random
import os
from datetime import timedelta


#BigQuery
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPICallError

fake = Faker('es_ES')


#He fijado estos números, para que todos obtengamos
#los mismos datos al ejecutar el notebook
random.seed(42)
Faker.seed(42)

#Carpeta de salida
OUTPUT_DIR = '../data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

#Configuración BigQuery
PROJECT_ID = "bytenova-495319"
DATASET_ID = "shopnow"            


print("Librerías cargadas correctamente")
print(f"Carpeta de salida: {os.path.abspath(OUTPUT_DIR)}")
print(f"BigQuery target   : {PROJECT_ID}.{DATASET_ID}")

Librerías cargadas correctamente
Carpeta de salida: c:\Users\pc\Documents\Team Challenge\SQL\Tc-sql-ByteNova\data
BigQuery target   : bytenova-495319.shopnow


## 2. Enumerados del modelo

In [2]:
#Valores
ACQUISITION_CHANNELS = ['organic', 'paid_ads', 'social_media', 'referral']
COUNTRIES = ['Spain', 'France', 'Germany', 'Italy', 'Portugal', 'Netherlands']

ORDER_STATUSES   = ['pending', 'confirmed', 'shipped', 'delivered', 'cancelled', 'returned']
PAYMENT_METHODS  = ['credit_card', 'debit_card', 'paypal', 'bank_transfer']
PAYMENT_STATUSES = ['completed', 'refunded', 'pending', 'failed']
RATING_VALUES    = [1, 2, 3, 4, 5]
DISCOUNT_VALUES  = [0.0, 0.05, 0.10, 0.15, 0.20]  

CATEGORY_NAMES = [
    'Smartphones', 'Laptops', 'Tablets', 'Headphones',
    'Cameras', 'Smart Home', 'Wearables', 'Gaming',
    'Accessories', 'Monitors'
]

CATEGORY_DESCS = {
    'Smartphones':  'Teléfonos inteligentes de última generación',
    'Laptops':      'Ordenadores portátiles para trabajo y ocio',
    'Tablets':      'Tabletas para productividad y entretenimiento',
    'Headphones':   'Auriculares y accesorios de audio',
    'Cameras':      'Cámaras digitales y de acción',
    'Smart Home':   'Dispositivos para el hogar inteligente',
    'Wearables':    'Relojes inteligentes y pulseras de actividad',
    'Gaming':       'Videojuegos, consolas y periféricos',
    'Accessories':  'Accesorios y complementos electrónicos',
    'Monitors':     'Monitores y pantallas para escritorio',
}

#Marcas por categoría
BRANDS_BY_CATEGORY = {
    'Smartphones':  ['Apple', 'Samsung', 'Xiaomi', 'Google', 'OnePlus'],
    'Laptops':      ['Apple', 'Dell', 'Lenovo', 'HP', 'Asus'],
    'Tablets':      ['Apple', 'Samsung', 'Lenovo', 'Microsoft', 'Huawei'],
    'Headphones':   ['Sony', 'Bose', 'Jabra', 'Sennheiser', 'Apple'],
    'Cameras':      ['Canon', 'Nikon', 'Sony', 'GoPro', 'Fujifilm'],
    'Smart Home':   ['Philips', 'Google', 'Amazon', 'Xiaomi', 'Ikea'],
    'Wearables':    ['Apple', 'Garmin', 'Fitbit', 'Samsung', 'Xiaomi'],
    'Gaming':       ['Sony', 'Microsoft', 'Nintendo', 'Razer', 'Logitech'],
    'Accessories':  ['Anker', 'Belkin', 'Logitech', 'Ugreen', 'Baseus'],
    'Monitors':     ['LG', 'Samsung', 'Dell', 'Asus', 'BenQ'],
}

print("Categorías definidas")

Categorías definidas


## 3. Tabla `categories`

Sin FKs → se genera primero.  
Los `created_at` se distribuyen entre hace 4 y 2 años para que sean anteriores a los productos.


In [3]:
def generar_categorias() -> pd.DataFrame:
    categorias = []

    for nombre in CATEGORY_NAMES:
        categorias.append({
            'category_id': str(uuid.uuid4()),
            'name':        nombre,
            'description': CATEGORY_DESCS[nombre],
            #Rango amplio. Para que no sean todos del 2026
            'created_at':  fake.date_time_between(start_date='-4y', end_date='-2y'),
        })

    return pd.DataFrame(categorias)


df_categories = generar_categorias()
df_categories


,category_id,name,description,created_at
0,a630e39b-24f2-468a-a535-f2092d38ccb0,Smartphones,Teléfonos inteligentes de última generación,2023-09-14 12:33:39
1,9e8e995d-3b81-4037-936b-48b15c57f72f,Laptops,Ordenadores portátiles para trabajo y ocio,2022-07-31 08:08:34
2,b7b879d1-e9e4-4e9d-bc2e-b976aa3cc7b0,Tablets,Tabletas para productividad y entretenimiento,2022-05-25 07:00:56
3,2028915d-5f18-421a-9fad-0bc02dc94785,Headphones,Auriculares y accesorios de audio,2023-12-02 20:17:04
4,4e3e1320-0f30-4be4-bdc1-fe5b7f346316,Cameras,Cámaras digitales y de acción,2022-12-05 11:41:58
5,357c4eea-ab79-4a9d-9500-5b0f56aa34f9,Smart Home,Dispositivos para el hogar inteligente,2022-11-12 01:53:47
6,e3511aef-c399-40e4-80ef-e02daa90e006,Wearables,Relojes inteligentes y pulseras de actividad,2022-10-26 05:43:52
7,be20e078-b8ec-4d7c-b2bb-b5e53a5a4b92,Gaming,"Videojuegos, consolas y periféricos",2022-08-22 05:57:24
8,e8b6943b-78f5-4a80-9188-ebc7785f7ca2,Accessories,Accesorios y complementos electrónicos,2023-11-28 21:35:59
9,d52b9b30-a81f-4425-aa2c-d8f2daf3150a,Monitors,Monitores y pantallas para escritorio,2022-07-24 11:25:47


## 4. Tabla `products`

FK → `categories.category_id`

**Decisiones de diseño:**
- `price` > `cost` siempre (margen 20–80%) — garantizado por construcción
- `price` refleja el precio **actual**; el precio de venta histórico va en `order_items.unit_price`
- Marca elegida según categoría del producto (coherencia de catálogo)


In [4]:
def generar_productos(df_cats: pd.DataFrame, n: int = 70) -> pd.DataFrame:
    productos = []

    #category_id - nombre, para elegir la marca correcta
    cat_id_to_name = dict(zip(df_cats['category_id'], df_cats['name']))

    for _ in range(n):
        cat_id   = random.choice(df_cats['category_id'].tolist())
        cat_name = cat_id_to_name[cat_id]
        brand    = random.choice(BRANDS_BY_CATEGORY[cat_name])

        #PVP siempre > coste: margen entre 20 % y 80 %
        coste  = round(random.uniform(20.0, 800.0), 2)
        precio = round(coste * random.uniform(1.2, 1.8), 2)

        #Nombre realista
        model_code = fake.bothify('??-###').upper()
        name = f"{brand} {cat_name[:-1] if cat_name.endswith('s') else cat_name} {model_code}"

        productos.append({
            'product_id':  str(uuid.uuid4()),
            'category_id': cat_id,
            'name':        name,
            'description': fake.sentence(nb_words=10),
            'brand':       brand,
            'price':       precio,
            'cost':        coste,
            'stock':       random.randint(0, 400),
            'is_active':   random.choice([True, True, True, False]),  # 75 % activos
            #Productos más recientes
            'created_at':  fake.date_time_between(start_date='-2y', end_date='-6m'),
        })

    return pd.DataFrame(productos)


df_products = generar_productos(df_categories, n=70)
df_products.head()


,product_id,category_id,name,description,brand,price,cost,stock,is_active,created_at
0,ea2dbd62-e754-4b1d-a660-dee929125e83,9e8e995d-3b81-4037-936b-48b15c57f72f,Apple Laptop BC-819,Sí lado clase importancia vivir habrá.,Apple,806.02,598.41,71,True,2025-03-27 04:43:03
1,966115f1-33eb-4fd5-a6ab-b5594feb4d97,e8b6943b-78f5-4a80-9188-ebc7785f7ca2,Anker Accessorie RZ-379,Peso medio relaciones alguna acuerdo serán.,Anker,585.86,480.58,47,True,2024-07-23 17:38:20
2,ab01d1d5-c518-43c7-b1d9-4357c246ddc8,2028915d-5f18-421a-9fad-0bc02dc94785,Apple Headphone WW-161,Nombre había etc hecho material te luis pueda ...,Apple,752.35,489.57,366,False,2025-12-26 19:20:23
3,9224fab5-5a4e-4dfe-89d2-68251e82e03f,2028915d-5f18-421a-9fad-0bc02dc94785,Sennheiser Headphone GY-413,Grupos período sea igual grande precisamente h...,Sennheiser,808.49,479.63,3,True,2025-06-24 05:23:25
4,9dab23d4-7ca9-4eb6-b282-ed263d99f269,e3511aef-c399-40e4-80ef-e02daa90e006,Fitbit Wearable YR-327,Hizo sé llega yo informe puerto estoy uno huma...,Fitbit,314.67,236.74,390,True,2026-03-23 11:59:26


## 5. Tabla `customers`

Sin FKs → se genera en paralelo a `categories`.

**Decisiones de diseño (justificación 3NF):**  
`country` se almacena como STRING directamente en `customers`. 

No se crea una tabla `countries` separada porque no hay otros atributos que dependan de `country` — esto es válido en 3NF.

In [5]:
def generar_clientes(n: int = 500) -> pd.DataFrame:
    clientes = []

    for _ in range(n):
        clientes.append({
            'customer_id':         str(uuid.uuid4()),
            'first_name':          fake.first_name(),
            'last_name':           fake.last_name(),
            'email':               fake.unique.email(), 
            'phone':               fake.phone_number(),
            'city':                fake.city(),
            'country':             random.choice(COUNTRIES),
            'acquisition_channel': random.choice(ACQUISITION_CHANNELS),
            'registered_at':       fake.date_time_between(start_date='-2y', end_date='now'),
            'is_active':           random.choice([True, True, True, False]),  #75% activos
        })

    return pd.DataFrame(clientes)


df_customers = generar_clientes(500)
df_customers.head()

,customer_id,first_name,last_name,email,phone,city,country,acquisition_channel,registered_at,is_active
0,fa64ddde-9661-4a57-9c7f-a430243c216e,Ciriaco,Garay,ayusomiriam@example.com,+34880 293 183,Ceuta,Portugal,organic,2025-01-15 13:05:43,False
1,e6d52bcb-105e-4be9-a0da-e1ebdea0510b,María Ángeles,Coloma,lluchrosa-maria@example.org,+34 842842102,Huelva,Portugal,paid_ads,2026-01-08 22:40:58,True
2,5d9d2ed5-7a9b-414f-bb74-68f09bb4528e,Modesta,Gárate,alcides24@example.org,+34841 681 177,Málaga,Spain,referral,2025-08-09 11:02:17,True
3,60dd8bb4-485e-4165-aac0-ef19d69e0904,César,Sans,encarnacionaroca@example.net,+34984 47 00 76,Soria,Portugal,paid_ads,2024-07-28 04:51:20,True
4,2f90db11-4990-48bd-bf1c-b6b9330d9b28,Amor,Salcedo,berrocalmodesto@example.net,+34 845124998,Jaén,Italy,organic,2025-08-12 08:29:33,True


 ## 6. Tabla `orders`
FK → customers.customer_id



**Decisiones de diseño:**

`ordered_at` siempre ≥ `customer.registered_at` (integridad temporal)

`shipped_at` pertenece a [ordered_at + 1d, ordered_at + 5d]  — solo si no pending/cancelled

`delivered_at` pertenece a [shipped_at + 2d, shipped_at + 14d] — solo si delivered

Pesos de status realistas: 60 % delivered, resto distribuido

In [6]:
def generar_pedidos(df_custs: pd.DataFrame, n: int = 2000) -> pd.DataFrame:
    pedidos = []

    #Construimos lookup registered_at por customer_id para respetar temporalidad
    reg_lookup = dict(zip(df_custs['customer_id'], df_custs['registered_at']))
    customer_ids = df_custs['customer_id'].tolist()

    #status realistas: 60 % delivered para suficientes reviews
    status_pool = (
        ['delivered'] * 60 +
        ['shipped']   * 10 +
        ['confirmed'] * 10 +
        ['pending']   * 8  +
        ['cancelled'] * 8  +
        ['returned']  * 4
    )

    for _ in range(n):
        customer_id = random.choice(customer_ids)
        registered_at = reg_lookup[customer_id]

        #ordered_at siempre posterior al registro del cliente
        ordered_at = fake.date_time_between(
            start_date=registered_at,
            end_date='now',
        )

        status = random.choice(status_pool)

        shipped_at   = None
        delivered_at = None

        #shipped_at: pedidos que ya salieron del almacén
        if status in ('shipped', 'delivered', 'returned'):
            shipped_at = ordered_at + timedelta(days=random.randint(1, 5))

        #delivered_at: solo pedidos entregados o devueltos
        if status in ('delivered', 'returned'):
            delivered_at = shipped_at + timedelta(days=random.randint(2, 14))

        pedidos.append({
            'order_id':         str(uuid.uuid4()),
            'customer_id':      customer_id,
            'status':           status,
            'shipping_address': fake.street_address(),
            'shipping_city':    fake.city(),
            'shipping_country': random.choice(COUNTRIES),
            'ordered_at':       ordered_at,
            'shipped_at':       shipped_at,
            'delivered_at':     delivered_at,
        })

    return pd.DataFrame(pedidos)




df_orders = generar_pedidos(df_customers, n=2000)
display(df_orders.head())
print(f"orders     : {len(df_orders)} filas")

,order_id,customer_id,status,shipping_address,shipping_city,shipping_country,ordered_at,shipped_at,delivered_at
0,50326dc5-b2a3-4b39-92ad-b35930a467b7,a6ac3a47-fcb0-40d7-a175-e9d4ccd91e95,confirmed,Urbanización Sonia Juárez 95 Apt. 23,Cádiz,Portugal,2025-06-20 13:26:12,NaT,NaT
1,5ec6ce25-a5ca-459f-bf3d-3d3082dd2bf3,bf655a77-beb9-4f3f-b301-c132e4b8b12a,shipped,Vial Reynaldo Cantero 53,Ciudad,Spain,2026-05-04 07:31:38,2026-05-05 07:31:38,NaT
2,4ed1176f-94b2-49bc-b660-3019d4437068,68413979-557f-4121-9443-af51318cc6be,delivered,Pasadizo Nerea Roura 2 Apt. 23,Murcia,Italy,2025-09-10 19:57:41,2025-09-13 19:57:41,2025-09-27 19:57:41
3,c9eb0b60-7a61-4110-bbcc-a02c4774a4c9,5b164e20-35bb-43e0-ad89-1eeb189575a5,delivered,Ronda de Fanny Hidalgo 4 Puerta 4,Guadalajara,Portugal,2026-02-28 12:37:33,2026-03-02 12:37:33,2026-03-13 12:37:33
4,5b44bf55-1b51-4d2d-ad5a-b6863aeee61e,cef2df06-b149-4b80-8d9f-df2b810c24f3,delivered,Vial de Renata Verdejo 273,Zamora,France,2024-11-30 05:44:43,2024-12-05 05:44:43,2024-12-12 05:44:43


orders     : 2000 filas


##  7. Tabla `order_items`
FK → orders.order_id, products.product_id



**Decisiones de diseño:**

Media 2-3 productos por pedido

`unit_price` ≈ product.price ± 10 % (variación histórica de precio)

Descuento entre 0 y 30 % con probabilidad del 40 %

No se repite el mismo producto en el mismo pedido


In [7]:
def generar_lineas_pedido(
    df_ords: pd.DataFrame,
    df_prods: pd.DataFrame,
) -> pd.DataFrame:
    product_ids = df_prods['product_id'].tolist()
    price_lookup = dict(zip(df_prods['product_id'], df_prods['price']))
    lineas_pedido = []

    for _, order in df_ords.iterrows():
        #2-3 productos por pedido
        n_items = random.choices([2, 3, 4], weights=[50, 35, 15])[0]
        chosen_products = random.sample(product_ids, min(n_items, len(product_ids)))

        for prod_id in chosen_products:
            base_price = price_lookup[prod_id]
            #Variación histórica ±10 % respecto al precio actual
            unit_price = round(base_price * random.uniform(0.90, 1.10), 2)
            #Descuento discreto(40 % de probabilidad)
            discount = random.choice(DISCOUNT_VALUES[1:]) if random.random() < 0.40 else 0.0

            lineas_pedido.append({
                'order_item_id': str(uuid.uuid4()),
                'order_id':      order['order_id'],
                'product_id':    prod_id,
                'quantity':      random.randint(1, 4),
                'unit_price':    unit_price,
                'discount':      discount,
            })

    return pd.DataFrame(lineas_pedido)


df_order_items = generar_lineas_pedido(df_orders, df_products)
display(df_order_items.head())
print(f"order_items: {len(df_order_items)} filas")

,order_item_id,order_id,product_id,quantity,unit_price,discount
0,fed603b1-f873-46d1-ac06-a8e21286d3c0,50326dc5-b2a3-4b39-92ad-b35930a467b7,97511a5e-6099-4aa8-9cfd-3db29018883d,3,624.37,0.2
1,e9faf6ef-753f-4bf1-81ec-b98ee872cdde,50326dc5-b2a3-4b39-92ad-b35930a467b7,14f5b3d5-08d0-498f-bf11-a526274269a7,2,793.29,0.0
2,daaedd5c-2aa1-4bd8-ab7c-da68fc3132dc,5ec6ce25-a5ca-459f-bf3d-3d3082dd2bf3,2e8d7c93-7666-4983-937a-14a11ffcaded,4,1542.19,0.0
3,040dca63-45bd-4431-9123-af7a4aea93df,5ec6ce25-a5ca-459f-bf3d-3d3082dd2bf3,19e226bf-976a-4cca-a569-8c6d6223912b,1,260.32,0.1
4,bb156f30-b7b7-40df-954e-3b7957aeea06,4ed1176f-94b2-49bc-b660-3019d4437068,ead61adc-3c4f-4074-b84a-faba31ec1788,3,968.77,0.0


order_items: 5303 filas


## 8. Tabla `payments`
FK → orders.order_id  (1 pago por pedido)



**Decisiones de diseño:**

`amount` = suma de (unit_price * quantity * (1 - discount)) por pedido

`paid_at` pertenece a [ordered_at, ordered_at + 2d]

`pedidos cancelled/returned` → status refunded o failed

`pedidos delivered/shipped`  → status completed (mayoría)

In [8]:
def generar_pagos(
    df_ords: pd.DataFrame,
    df_items: pd.DataFrame,
) -> pd.DataFrame:

    #Total por pedido desde order_items
    df_items_calc = df_items.copy()
    df_items_calc['line_total'] = (
        df_items_calc['unit_price'] *
        df_items_calc['quantity'] *
        (1 - df_items_calc['discount'])
    )
    totals = df_items_calc.groupby('order_id')['line_total'].sum().round(2)

    payments = []
    for _, order in df_ords.iterrows():
        order_id   = order['order_id']
        status     = order['status']
        ordered_at = order['ordered_at']

        #Mapeo coherente de status de pedido a status de pago
        if status == 'cancelled':
            pay_status = random.choice(['failed', 'refunded'])
        elif status == 'returned':
            pay_status = 'refunded'
        elif status == 'pending':
            pay_status = 'pending'
        elif status == 'confirmed':
            #Pedido confirmado → pago completado pero aún no enviado
            pay_status = 'completed'
        else:
            #shipped, delivered
            pay_status = 'completed'

        paid_at = (
            ordered_at + timedelta(hours=random.randint(0, 48))
            if pay_status in ('completed', 'refunded')
            else None
        )

        payments.append({
            'payment_id': str(uuid.uuid4()),
            'order_id':   order_id,
            'method':     random.choice(PAYMENT_METHODS),
            'status':     pay_status,
            'amount':     totals.get(order_id, 0.0),
            'paid_at':    paid_at,
        })

    return pd.DataFrame(payments)


df_payments = generar_pagos(df_orders, df_order_items)
display(df_payments.head())
print(f"payments   : {len(df_payments)} filas")

,payment_id,order_id,method,status,amount,paid_at
0,dda56f54-5a15-40b2-9244-81f8f0ec852a,50326dc5-b2a3-4b39-92ad-b35930a467b7,debit_card,completed,3085.07,2025-06-21 08:26:12
1,c346063a-4ca5-43e4-a7dc-2f4981358c7e,5ec6ce25-a5ca-459f-bf3d-3d3082dd2bf3,credit_card,completed,6403.05,2026-05-04 13:31:38
2,a300d7ad-c59c-4a63-b6a9-d7ef39ccf9a7,4ed1176f-94b2-49bc-b660-3019d4437068,paypal,completed,6872.37,2025-09-11 21:57:41
3,bc272dce-df9a-4084-a849-ba7e1def583e,c9eb0b60-7a61-4110-bbcc-a02c4774a4c9,credit_card,completed,6344.18,2026-02-28 16:37:33
4,924d51c7-76e3-4076-9b51-6c24452ea646,5b44bf55-1b51-4d2d-ad5a-b6863aeee61e,paypal,completed,3601.68,2024-11-30 06:44:43


payments   : 2000 filas


## 9. Tabla `reviews`
FK → order_items.order_item_id


Solo `order_items` de pedidos con status 'delivered'

In [9]:
def generar_valoraciones(
    df_ords: pd.DataFrame,
    df_items: pd.DataFrame,
    tasa: float = 0.35,
) -> pd.DataFrame:
    """
    Genera una review por order_item (relación 1:1 según el diseño de la Data Modeler).
    Solo se valoran order_items de pedidos con status 'delivered'.
    Se muestrea el 35 % de esas líneas.
    """
    #Pedidos entregados con su delivered_at
    delivered = df_ords[df_ords['status'] == 'delivered'][['order_id', 'delivered_at']]
    items_delivered = df_items.merge(delivered, on='order_id', how='inner')

    #Muestreo del 35 % (sin reemplazo → garantiza unicidad: 1 review por order_item)
    muestra = items_delivered.sample(frac=tasa, random_state=42)

    #Distribución de rating según RATING_VALUES = [1,2,3,4,5]
    rating_pool = [1, 2, 3, 3, 4, 4, 4, 5, 5, 5]

    reviews = []
    for _, item in muestra.iterrows():
        delivered_at = item['delivered_at']
        created_at   = delivered_at + timedelta(days=random.randint(1, 30))

        reviews.append({
            'review_id':     str(uuid.uuid4()),
            'order_item_id': item['order_item_id'],
            'rating':        random.choice(rating_pool),
            'comment':       fake.sentence(nb_words=random.randint(6, 18)) if random.random() > 0.3 else None,
            'created_at':    created_at,
        })

    return pd.DataFrame(reviews)


df_reviews = generar_valoraciones(df_orders, df_order_items, tasa=0.35)
display(df_reviews.head())
print(f"reviews    : {len(df_reviews)} filas")

,review_id,order_item_id,rating,comment,created_at
0,8c05977c-2039-43ce-82af-f591c365c5b2,ee839016-3214-420e-8c62-c1cb87e05426,3,Buscar tengo mucho volvió haciendo quienes pos...,2026-05-21 17:44:49
1,c764889a-adb1-4cc0-b23c-bd7c91e5d0b7,1a5cccd9-e6a5-4197-af35-cd789c001cd8,3,Tratamiento dejó hace señor cada mismo demasiado.,2026-04-29 13:41:07
2,2d181868-669e-46eb-96c8-b4cbabb8210a,1faeb00a-a355-4f17-b08f-a6c017f839fd,1,Cuales plazo arriba sean formas esa encuentran...,2026-05-18 04:01:23
3,6b30c660-3fd8-436a-b8d2-d1d03c81244a,161f7438-f14b-4738-a485-b9d2ada008ff,3,Adelante iglesia zona pedro porque análisis su...,2025-01-28 21:41:09
4,3a061d23-c37d-4d11-a396-1320fe0e4480,5065ef0a-710a-465a-b1e7-3461db9003af,4,NaN,2025-11-17 10:08:15


reviews    : 1075 filas


## 10. Verificaciones de calidad

Comprobaciones rápidas antes de exportar para detectar errores antes de cargar en BigQuery.

In [10]:
print("\n" + "=" * 55)
print("RESUMEN DE TABLAS GENERADAS")
print("=" * 55)
tablas = {
    'categories':  df_categories,
    'products':    df_products,
    'customers':   df_customers,
    'orders':      df_orders,
    'order_items': df_order_items,
    'payments':    df_payments,
    'reviews':     df_reviews,
}
for nombre, df in tablas.items():
    print(f"  {nombre:<14}: {len(df):>6} filas")

print("\nIntegridad referencial")

#Precio > coste
assert (df_products['price'] > df_products['cost']).all(), "FALLO: price debe ser > cost"
print("✓ Precio > coste en todos los productos")

#Emails sin duplicados
assert df_customers['email'].nunique() == len(df_customers), "FALLO: emails duplicados"
print("✓ Emails sin duplicados en customers")

#Productos → categorias
ids_cats = set(df_categories['category_id'])
assert df_products['category_id'].isin(ids_cats).all(), "FALLO: products → categories"
print("✓ FK productos → categorias")

#Orders → customers
ids_custs = set(df_customers['customer_id'])
assert df_orders['customer_id'].isin(ids_custs).all(), "FALLO: orders → customers"
print("✓ FK orders → customers")

#Order_items → orders
ids_orders = set(df_orders['order_id'])
assert df_order_items['order_id'].isin(ids_orders).all(), "FALLO: order_items → orders"
print("✓ order_items → orders")

#Order_items → products
ids_prods = set(df_products['product_id'])
assert df_order_items['product_id'].isin(ids_prods).all(), "FALLO: order_items → products"
print("✓ order_items → products")

#Payments → orders
assert df_payments['order_id'].isin(ids_orders).all(), "FALLO: payments → orders"
assert df_payments['order_id'].nunique() == len(df_payments), "FALLO: más de un pago por pedido"
print("✓ payments → orders (1 pago por pedido)")

#Reviews → order_items
ids_items = set(df_order_items['order_item_id'])
assert df_reviews['order_item_id'].isin(ids_items).all(), "FALLO: reviews → order_items"
print("✓ reviews → order_items")

print("\nMínimos del enunciado")
assert len(df_categories) == 10,    "FALLO: se necesitan exactamente 10 categorías"
assert len(df_products)   >= 70,    "FALLO: se necesitan ≥ 70 productos"
assert len(df_customers)  >= 500,   "FALLO: se necesitan ≥ 500 clientes"
assert len(df_orders)     >= 2000,  "FALLO: se necesitan ≥ 2000 pedidos"
assert len(df_order_items) >= 4000, "FALLO: se esperan ~4500 líneas de pedido"
print("Todos los mínimos del enunciado cumplidos")

print("\nTodas las verificaciones superadas — listos para exportar y cargar\n")


RESUMEN DE TABLAS GENERADAS
  categories    :     10 filas
  products      :     70 filas
  customers     :    500 filas
  orders        :   2000 filas
  order_items   :   5303 filas
  payments      :   2000 filas
  reviews       :   1075 filas

Integridad referencial
✓ Precio > coste en todos los productos
✓ Emails sin duplicados en customers
✓ FK productos → categorias
✓ FK orders → customers
✓ order_items → orders
✓ order_items → products
✓ payments → orders (1 pago por pedido)
✓ reviews → order_items

Mínimos del enunciado
Todos los mínimos del enunciado cumplidos

Todas las verificaciones superadas — listos para exportar y cargar



## 11. Exportar CSVs locales (para tener respaldo)

In [11]:
for nombre, df in tablas.items():
    path = os.path.join(OUTPUT_DIR, f"{nombre}.csv")
    df.to_csv(path, index=False)
    print(f"CSV exportado: {path}")

CSV exportado: ../data\categories.csv
CSV exportado: ../data\products.csv
CSV exportado: ../data\customers.csv
CSV exportado: ../data\orders.csv
CSV exportado: ../data\order_items.csv
CSV exportado: ../data\payments.csv
CSV exportado: ../data\reviews.csv


## 12. Carga en BigQuery

In [12]:
SCHEMAS: dict[str, list[bigquery.SchemaField]] = {
    'categories': [
        bigquery.SchemaField('category_id',  'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('name',          'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('description',   'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('created_at',    'TIMESTAMP', mode='NULLABLE'),
    ],
    'products': [
        bigquery.SchemaField('product_id',   'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('category_id',  'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('name',          'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('description',   'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('brand',         'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('price',         'FLOAT64',   mode='REQUIRED'),
        bigquery.SchemaField('cost',          'FLOAT64',   mode='REQUIRED'),
        bigquery.SchemaField('stock',         'INT64',     mode='NULLABLE'),
        bigquery.SchemaField('is_active',     'BOOL',      mode='NULLABLE'),
        bigquery.SchemaField('created_at',    'TIMESTAMP', mode='NULLABLE'),
    ],
    'customers': [
        bigquery.SchemaField('customer_id',          'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('first_name',            'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('last_name',             'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('email',                 'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('phone',                 'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('city',                  'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('country',               'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('acquisition_channel',   'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('registered_at',         'TIMESTAMP', mode='REQUIRED'),
        bigquery.SchemaField('is_active',             'BOOL',      mode='NULLABLE'),
    ],
    'orders': [
        bigquery.SchemaField('order_id',          'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('customer_id',        'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('status',             'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('shipping_address',   'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('shipping_city',      'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('shipping_country',   'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('ordered_at',         'TIMESTAMP', mode='REQUIRED'),
        bigquery.SchemaField('shipped_at',         'TIMESTAMP', mode='NULLABLE'),
        bigquery.SchemaField('delivered_at',       'TIMESTAMP', mode='NULLABLE'),
    ],
    'order_items': [
        bigquery.SchemaField('order_item_id', 'STRING',  mode='REQUIRED'),
        bigquery.SchemaField('order_id',       'STRING',  mode='REQUIRED'),
        bigquery.SchemaField('product_id',     'STRING',  mode='REQUIRED'),
        bigquery.SchemaField('quantity',        'INT64',   mode='REQUIRED'),
        bigquery.SchemaField('unit_price',      'FLOAT64', mode='REQUIRED'),
        bigquery.SchemaField('discount',        'FLOAT64', mode='NULLABLE'),
    ],
    'payments': [
        bigquery.SchemaField('payment_id', 'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('order_id',    'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('method',      'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('status',      'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('amount',      'FLOAT64',   mode='REQUIRED'),
        bigquery.SchemaField('paid_at',     'TIMESTAMP', mode='NULLABLE'),
    ],
    'reviews': [
        bigquery.SchemaField('review_id',      'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('order_item_id',   'STRING',    mode='REQUIRED'),
        bigquery.SchemaField('rating',          'INT64',     mode='REQUIRED'),
        bigquery.SchemaField('comment',         'STRING',    mode='NULLABLE'),
        bigquery.SchemaField('created_at',      'TIMESTAMP', mode='NULLABLE'),
    ],
}


def cargar_tabla_bigquery(
    df: pd.DataFrame,
    tabla: str,
    client: bigquery.Client,
    project_id: str = PROJECT_ID,
    dataset_id: str = DATASET_ID,
    write_disposition: str = 'WRITE_TRUNCATE',
) -> bool:
    """
    Carga un DataFrame en BigQuery.

    Parámetros
    ----------
    df               : DataFrame a cargar.
    tabla            : Nombre de la tabla destino (sin dataset ni proyecto).
    client           : Cliente BigQuery autenticado.
    project_id       : ID del proyecto GCP.
    dataset_id       : ID del dataset BigQuery.
    write_disposition: 'WRITE_TRUNCATE' (sobreescribir) o 'WRITE_APPEND'.

    Retorna
    -------
    True si la carga fue exitosa, False en caso de error.
    """
    table_ref = f"{project_id}.{dataset_id}.{tabla}"
    schema    = SCHEMAS.get(tabla)

    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=write_disposition,
        #Permite columnas nulas que no están en el DataFrame
        ignore_unknown_values=False,
    )

    print(f"\n→ Cargando '{tabla}' ({len(df)} filas) en {table_ref} …")
    try:
        job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
        job.result()

        #Validación post-carga
        tabla_bq  = client.get_table(table_ref)
        filas_bq  = tabla_bq.num_rows
        filas_df  = len(df)

        if filas_bq == filas_df:
            print(f"'{tabla}' cargada correctamente ({filas_bq} filas en BQ)")
            return True
        else:
            print(f"'{tabla}': DataFrame tiene {filas_df} filas pero BQ reporta {filas_bq}")
            return False

    except GoogleAPICallError as e:
        print(f"Error de API en '{tabla}': {e}")
        return False
    except Exception as e:
        print(f"Error inesperado en '{tabla}': {e}")
        return False


def cargar_todas_las_tablas(
    tablas_dict: dict,
    project_id: str = PROJECT_ID,
    dataset_id: str = DATASET_ID,
) -> dict[str, bool]:
    
    load_dotenv()
    
    """
    Carga todas las tablas en BigQuery respetando el orden de dependencias
    (primero maestras sin FK, luego transaccionales).

    Retorna {nombre_tabla: bool} con el resultado de cada carga.
    """
    #Orden según FKs del diagrama
    orden_carga = [
        'categories',   # sin FK
        'customers',    # sin FK
        'products',     # FK → categories
        'orders',       # FK → customers
        'order_items',  # FK → orders, products
        'payments',     # FK → orders
        'reviews',      # FK → order_items
    ]

    print("\n" + "=" * 55)
    print("INICIO DE CARGA EN BIGQUERY")
    print("=" * 55)
    print(f"Proyecto : {project_id}")
    print(f"Dataset  : {dataset_id}")

    try:
        client = bigquery.Client(project=project_id)
        print("Cliente BigQuery inicializado correctamente\n")
    except Exception as e:
        print(f"No se pudo crear el cliente BigQuery: {e}")
        print("Asegúrate de haber autenticado con:")
        print("gcloud auth application-default login")
        return {t: False for t in orden_carga}

    resultados = {}
    for tabla in orden_carga:
        if tabla not in tablas_dict:
            print(f"Tabla '{tabla}' no encontrada en tablas_dict — omitida")
            resultados[tabla] = False
            continue
        resultados[tabla] = cargar_tabla_bigquery(
            df=tablas_dict[tabla],
            tabla=tabla,
            client=client,
            project_id=project_id,
            dataset_id=dataset_id,
        )

    #Resumen final
    print("\n" + "=" * 55)
    print("RESUMEN DE CARGA")
    print("=" * 55)
    exitosas  = sum(v for v in resultados.values())
    fallidas  = len(resultados) - exitosas
    for tabla, ok in resultados.items():
        estado = "OK" if ok else "FALLO"
        print(f"  {tabla:<14} {estado}")
    print(f"\nTablas cargadas: {exitosas}/{len(resultados)}  |  Fallos: {fallidas}")

    if fallidas == 0:
        print("\nTodas las tablas cargadas exitosamente en BigQuery")
    else:
        print("\nRevisa los errores anteriores y vuelve a ejecutar las tablas fallidas")

    return resultados


## 13. Ejecutar carga

In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

creds = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
print(f"Ruta en .env      : {creds}")
print(f"¿Archivo existe?  : {os.path.exists(creds) if creds else 'NO - variable vacía'}")
print(f"PROJECT_ID        : {os.getenv('GCP_PROJECT_ID')}")
print(f"DATASET_ID        : {os.getenv('BQ_DATASET_ID')}")

Ruta en .env      : C:/Users/pc/Documents/Team Challenge/SQL/Tc-sql-ByteNova/credentials/service-account.json
¿Archivo existe?  : True
PROJECT_ID        : bytenova-495319
DATASET_ID        : shopnow


In [14]:
#Requiere autenticación GCP activa

resultados_carga = cargar_todas_las_tablas(tablas)


INICIO DE CARGA EN BIGQUERY
Proyecto : bytenova-495319
Dataset  : shopnow
Cliente BigQuery inicializado correctamente


→ Cargando 'categories' (10 filas) en bytenova-495319.shopnow.categories …


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


'categories' cargada correctamente (10 filas en BQ)

→ Cargando 'customers' (500 filas) en bytenova-495319.shopnow.customers …


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


'customers' cargada correctamente (500 filas en BQ)

→ Cargando 'products' (70 filas) en bytenova-495319.shopnow.products …


...

'order_items' cargada correctamente (5303 filas en BQ)

→ Cargando 'payments' (2000 filas) en bytenova-495319.shopnow.payments …


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


'payments' cargada correctamente (2000 filas en BQ)

→ Cargando 'reviews' (1075 filas) en bytenova-495319.shopnow.reviews …


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


'reviews' cargada correctamente (1075 filas en BQ)

RESUMEN DE CARGA
  categories     OK
  customers      OK
  products       OK
  orders         OK
  order_items    OK
  payments       OK
  reviews        OK

Tablas cargadas: 7/7  |  Fallos: 0

Todas las tablas cargadas exitosamente en BigQuery


## 14. Recarga individual (por si falla solo una tabla)

In [15]:
print("\nScript completado. Para cargar en BigQuery:")
print("1. Ajusta PROJECT_ID y DATASET_ID al inicio del script")
print("2. Autentica: gcloud auth application-default login")
print("3. Descomenta la línea con cargar_todas_las_tablas(tablas)")


Script completado. Para cargar en BigQuery:
1. Ajusta PROJECT_ID y DATASET_ID al inicio del script
2. Autentica: gcloud auth application-default login
3. Descomenta la línea con cargar_todas_las_tablas(tablas)
